---
title: "Showcase & tests"
---

An animated [Turtle graphics](https://docs.python.org/3/library/turtle.html) widget for Jupyter, built on [`anywidget`](https://anywidget.dev) so it renders identically in
VS Code, JupyterLab, Notebook 7, and Colab. Each cell below draws something and animates it on an HTML canvas. Pass `show_code=True` to display the cell's source beside the canvas and highlight each line in sync with the drawing. Use the Replay/Pause buttons under any canvas.

In [1]:
from turtle_widget import Turtle

## 1. The basics

A turtle starts at the centre `(0, 0)` facing **east** (heading `0`). `forward`/`backward`
move it, `left`/`right` rotate it (degrees, counter-clockwise positive). Here's a square.

In [2]:
t = Turtle()
t.speed(6)
for _ in range(4):
    t.forward(120)
    t.left(90)

## 2. Synced code highlighting (`show_code=True`)

This is your original spiral. With `show_code=True`, the source appears on the right and
the active line is highlighted as the animation reaches it.

In [3]:
t = Turtle(show_code=True)
t.speed(1)
colours = ["red", "blue", "yellow", "brown", "black", "purple", "green"]

t.penup(); t.left(90); t.forward(200); t.right(90); t.pendown()
for i in range(0, 18):
    t.pencolor(colours[i % 7])
    t.right(20)
    t.forward(50)

t.right(180)
t.home()

## 3. Colors, pen width & fills

Colours accept CSS names (`"gold"`), hex (`"#ff8800"`), or RGB tuples (`(0.1, 0.5, 0.9)`
on the 0–1 scale, or `(255, 128, 0)` on the 0–255 scale). Wrap a path in
`begin_fill()` / `end_fill()` to fill it — the fill lands *behind* the outline, just like
real turtle.

In [4]:
t = Turtle()
t.speed(8)
t.pensize(3)
t.pencolor("#d2691e")          # outline
t.fillcolor((1.0, 0.84, 0.0))  # gold, on the 0-1 RGB scale
t.begin_fill()
for _ in range(5):             # classic 5-point star
    t.forward(220)
    t.right(144)
t.end_fill()

## 4. Circles & arcs

`circle(radius)` draws a full circle to the **left** of the turtle; a negative radius goes
right. `circle(radius, extent)` draws only part of it. Stacking rotated circles gives a
spirograph.

In [6]:
t = Turtle()
t.speed(10)
palette = ["crimson", "darkorange", "seagreen", "royalblue", "purple", "teal"]
for i in range(12):
    t.pencolor(palette[i % len(palette)])
    t.circle(90)
    t.right(30)

## 5. Dots, stamps, text & jumps

`penup()` / `pendown()` lift and drop the pen so you can jump without drawing. `dot(size)`
plants a filled dot, `stamp()` leaves a copy of the turtle, and `write(text)` prints a
label.

In [7]:
t = Turtle(width=520, height=320)
t.speed(7)

t.penup(); t.goto(-210, 30); t.pendown()
for i, c in enumerate(["red", "orange", "green", "blue", "purple"]):
    t.dot(16 + i * 8, c)
    t.penup(); t.forward(90); t.pendown()

t.penup(); t.goto(-210, 90); t.pendown()
t.pencolor("black")
t.write("dots grow to the right", font=("sans-serif", 16, "normal"))

t.penup(); t.goto(0, -70); t.setheading(60); t.pendown()
t.pencolor("seagreen")
t.stamp()

34

## 6. Speed & instant mode

`speed(1)` is slowest, `speed(10)` is fastest, and `speed(0)` disables animation entirely
— the whole drawing appears at once, which is handy for dense figures. (Word forms also
work: `"slowest"`, `"slow"`, `"normal"`, `"fast"`, `"fastest"`.)

In [ ]:
t = Turtle()
#t.speed(0)               # no animation: drawn instantly
t.speed(10)
t.pencolor("teal")
t.pensize(1)
for _ in range(72):
    t.forward(160)
    t.left(175)          # a dense star spiral
t.hideturtle()

## 7. Reading turtle state

Getters return the current state without drawing anything: `pos()`, `xcor()`, `ycor()`,
`heading()`, `isdown()`, `isvisible()`. (Use `autoshow=False` when you only want to
inspect state and not render a widget.)

In [13]:
probe = Turtle(autoshow=False)
probe.forward(100); probe.left(90); probe.forward(50)
print("position:", probe.pos())
print("heading :", probe.heading())
print("pen down:", probe.isdown())

position: (100.0, 50.0)
heading : 90.0
pen down: True


## 8. Self-test (headless)

Geometry is computed in Python, so it can be validated without a browser. This cell asserts
a few invariants and checks the event stream is JSON-serializable (which is what gets sent
to the frontend).

In [ ]:
import json
from turtle_widget import Turtle

# movement + heading
t = Turtle(autoshow=False)
t.forward(100); t.left(90); t.forward(100)
assert abs(t.xcor() - 100) < 1e-9, t.xcor()
assert abs(t.ycor() - 100) < 1e-9, t.ycor()
assert round(t.heading() % 360, 6) == 90

# home returns to origin, heading 0
t.home()
assert abs(t.xcor()) < 1e-9 and abs(t.ycor()) < 1e-9
assert round(t.heading() % 360, 6) == 0

# circle returns (approximately) to its start
c = Turtle(autoshow=False)
c.begin_fill(); c.circle(60); c.end_fill()
assert abs(c.xcor()) < 1e-6 and abs(c.ycor()) < 1e-6
assert any(e["op"] == "fill" for e in c._events)

# every event is JSON-serializable and carries a source line + speed
payload = json.dumps(t._events)
assert all("line" in e and "speed" in e for e in t._events)

print("All self-tests passed.")
print("square+home events:", len(t._events), "| circle events:", len(c._events))